# ACFT Kaggle 01 - Generate Chunks and Publish

Restores canonical source datasets into `/kaggle/working/acft_data`, runs existing repo stages with resumable state, and versions generated chunk/manifests as a private Kaggle Dataset.

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

OWNER = os.environ.get("KAGGLE_OWNER", "drsriharshaguthik")
PROFILE = os.environ.get("PROFILE", "smoke")
RUN_TAG = os.environ.get("RUN_TAG", datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S"))
REPO_ROOT = Path(os.environ.get("ACFT_REPO_ROOT", "/kaggle/working/whisper-acft"))
WORK_ROOT = Path(os.environ.get("ACFT_WORK_ROOT", "/kaggle/working/acft_data"))
RUN_ROOT = Path(os.environ.get("ACFT_RUN_ROOT", "/kaggle/working/acft_runs")) / RUN_TAG
STATE_DIR = RUN_ROOT / "state"
CHUNKS_DIR = WORK_ROOT / "Record_chunks"
PUBLISH_AFTER_EACH_STAGE = os.environ.get("PUBLISH_AFTER_EACH_STAGE", "1") == "1"
DRY_RUN_PUBLISH = os.environ.get("DRY_RUN_PUBLISH", "0") == "1"
DRY_RUN_STAGES = os.environ.get("DRY_RUN_STAGES", "0") == "1"
SMOKE_MAX_TRANSCRIPTS = int(os.environ.get("SMOKE_MAX_TRANSCRIPTS", "25"))

RUN_ROOT.mkdir(parents=True, exist_ok=True)
STATE_DIR.mkdir(parents=True, exist_ok=True)
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

print({"profile": PROFILE, "run_tag": RUN_TAG, "repo_root": str(REPO_ROOT), "work_root": str(WORK_ROOT)})

In [ ]:
# Optional clone if the repo is not already attached/copied into Kaggle working.
if not (REPO_ROOT / "tools" / "kaggle_acft_helpers.py").exists():
    git_url = os.environ.get("ACFT_GIT_URL", "")
    if git_url:
        subprocess.run(["git", "clone", git_url, str(REPO_ROOT)], check=True)
    else:
        raise FileNotFoundError(
            f"Repo not found at {REPO_ROOT}. Attach repo files or set ACFT_GIT_URL."
        )

sys.path.insert(0, str(REPO_ROOT))
from tools import kaggle_acft_helpers as kh

print("helper", kh.__file__)

In [ ]:
def maybe_publish(stage_name: str) -> dict:
    if not PUBLISH_AFTER_EACH_STAGE:
        return {"stage": stage_name, "published": False}
    handle = kh.make_dataset_handle(OWNER, "acft-kaggle-chunks", RUN_TAG, PROFILE)
    kh.write_dataset_metadata(
        local_dir=CHUNKS_DIR,
        handle=handle,
        title=f"ACFT Kaggle chunks {RUN_TAG}",
        subtitle=f"{PROFILE} profile; latest completed stage: {stage_name}",
        keywords=["acft", "whisper", "chunks", "resumable"],
        licenses=[{"name": "unknown"}],
    )
    result = kh.publish_dataset(
        local_dir=CHUNKS_DIR,
        handle=handle,
        version_notes=f"{RUN_TAG}: {stage_name}",
        dry_run=DRY_RUN_PUBLISH,
    )
    print(json.dumps(result, indent=2, sort_keys=True))
    return result


def run_stage(stage_name: str, command: list[str], inputs: list[Path], outputs: list[Path], config: dict) -> dict:
    result = kh.run_resumable_stage(
        stage_name,
        command,
        inputs=inputs,
        outputs=outputs,
        config=config,
        state_dir=STATE_DIR,
        dry_run=DRY_RUN_STAGES,
    )
    print(json.dumps(result, indent=2, sort_keys=True))
    if not result.get("dry_run"):
        maybe_publish(stage_name)
    return result

In [ ]:
export_doc = REPO_ROOT / "docs" / "KAGGLE_PRIMARY_TRAINING_DATA_EXPORT.md"
handles = kh.parse_canonical_dataset_handles(export_doc)
result = kh.reconstruct_kaggle_sources(
    input_root=Path("/kaggle/input"),
    output_root=WORK_ROOT,
    dataset_handles=handles,
)
print(json.dumps(kh.as_jsonable_dataclass(result), indent=2, sort_keys=True))

sig = kh.stage_signature(inputs=[Path(result.inventory_path)], config={"handles": handles, "profile": PROFILE})
kh.mark_stage_done("source-reconstruct", outputs=[Path(result.inventory_path)], signature=sig, state_dir=STATE_DIR)
maybe_publish("source-reconstruct")

In [ ]:
transcript_dir = WORK_ROOT / "Transcriptions_corrected"
if PROFILE == "smoke":
    smoke_transcripts = RUN_ROOT / "smoke_transcripts"
    smoke_transcripts.mkdir(parents=True, exist_ok=True)
    copied = 0
    for src in sorted(transcript_dir.glob("*.json")):
        if "stage_0_apply_corrections_from_report_conflicts" in src.name:
            continue
        dst = smoke_transcripts / src.name
        if not dst.exists():
            shutil.copy2(src, dst)
        copied += 1
        if copied >= SMOKE_MAX_TRANSCRIPTS:
            break
    transcript_dir = smoke_transcripts
    print({"smoke_transcripts": copied, "dir": str(transcript_dir)})

stage1_cmd = [
    sys.executable,
    str(REPO_ROOT / "stage_1_Manifest_creation_local_only.py"),
    "--transcript-dir", str(transcript_dir),
    "--chunks-dir", str(CHUNKS_DIR),
    "--audio-source-dir", str(WORK_ROOT / "Record_harsha"),
    "--filename_policy", "kaggle_safe",
]
run_stage(
    "stage-01-manifest-plan",
    stage1_cmd,
    inputs=[transcript_dir, WORK_ROOT / "_kaggle_source_inventory.jsonl"],
    outputs=[CHUNKS_DIR / "pairs_pending.jsonl", CHUNKS_DIR / "tasks_pending.jsonl"],
    config={"profile": PROFILE, "script": "stage_1_Manifest_creation_local_only.py"},
)

In [ ]:
stage2_cmd = [
    sys.executable,
    str(REPO_ROOT / "stage_2_chunk_transcripts_sentence_parallel.py"),
    "--tasks_pending_path", str(CHUNKS_DIR / "tasks_pending.jsonl"),
    "--pairs_pending_path", str(CHUNKS_DIR / "pairs_pending.jsonl"),
    "--out_pairs_path", str(CHUNKS_DIR / "pairs_manifest.jsonl"),
    "--remaining_tasks_path", str(CHUNKS_DIR / "tasks_remaining.jsonl"),
    "--ffmpeg_workers", "2" if PROFILE == "smoke" else "8",
    "--task_workers", "2" if PROFILE == "smoke" else "24",
    "--save_every", "5" if PROFILE == "smoke" else "250",
    "--skip_existing", "1",
]
run_stage(
    "stage-02-cut-chunks",
    stage2_cmd,
    inputs=[CHUNKS_DIR / "tasks_pending.jsonl", CHUNKS_DIR / "pairs_pending.jsonl"],
    outputs=[CHUNKS_DIR / "pairs_manifest.jsonl", CHUNKS_DIR / "stage2_cut_state.json"],
    config={"profile": PROFILE, "script": "stage_2_chunk_transcripts_sentence_parallel.py"},
)

In [ ]:
train_manifest = CHUNKS_DIR / "pairs_manifest_stage13_train.jsonl"
test_manifest = CHUNKS_DIR / "pairs_manifest_stage13_test.jsonl"
stage13_cmd = [
    sys.executable,
    str(REPO_ROOT / "stage_13_group_split_train_test.py"),
    "--input_manifest", str(CHUNKS_DIR / "pairs_manifest.jsonl"),
    "--train_manifest", str(train_manifest),
    "--test_manifest", str(test_manifest),
    "--test_ratio", "0.10",
    "--seed", "1337",
]
run_stage(
    "stage-13-group-split",
    stage13_cmd,
    inputs=[CHUNKS_DIR / "pairs_manifest.jsonl"],
    outputs=[train_manifest, test_manifest],
    config={"profile": PROFILE, "script": "stage_13_group_split_train_test.py"},
)

randomized_manifest = CHUNKS_DIR / "pairs_manifest_stage15_train_randomized.jsonl"
stage15_cmd = [
    sys.executable,
    str(REPO_ROOT / "stage_15_b_advanced_randomize_manifest.py"),
    "--input_manifest", str(train_manifest),
    "--output_manifest", str(randomized_manifest),
    "--seed", "1337",
    "--validate_randomization",
]
run_stage(
    "stage-15-randomize-train",
    stage15_cmd,
    inputs=[train_manifest],
    outputs=[randomized_manifest],
    config={"profile": PROFILE, "script": "stage_15_b_advanced_randomize_manifest.py"},
)

print("Attach the published chunks dataset to notebook 02.")
print("train_manifest", randomized_manifest)
print("test_manifest", test_manifest)